# THUNAI Disease Training in Colab

Single notebook for:
- Dataset download and split
- Custom CNN training
- Pretrained transfer training
- Optional Hugging Face model training
- Artifact packaging for THUNAI

## Step 1: Configure run settings

Edit values if needed, then run this cell.

In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/SBH1928/THUNAI.AI.git"
REPO_BRANCH = "main"
USE_GOOGLE_DRIVE = False

HF_DATASET_ID = "dpdl-benchmark/plant_village"
HF_SPLIT = "train"
HF_IMAGE_COLUMN = "image"
HF_LABEL_COLUMN = "label"

VAL_SPLIT = 0.2
TEST_SPLIT = 0.1
SEED = 42

TRAIN_CUSTOM_CNN = True
TRAIN_TRANSFER_CNN = True
TRAIN_HF_MODEL = True

CUSTOM_EPOCHS = 20
TRANSFER_EPOCHS = 12
BATCH_SIZE = 32
LEARNING_RATE_CUSTOM = 0.001
LEARNING_RATE_TRANSFER = 0.0005

HF_MODEL_ID = "google/vit-base-patch16-224"
HF_EPOCHS = 6
HF_BATCH_SIZE = 16
HF_LEARNING_RATE = 5e-5

ARTIFACTS_ZIP_NAME = "thunai_disease_model_artifacts.zip"

print("Configuration loaded")

## Step 2: Optional Google Drive mount

In [ ]:
if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Google Drive mounted")
else:
    print("Drive mount skipped")

## Step 3: Clone repository and locate ML folder

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path


def find_ml_dir(repo_root: Path):
    candidates = [
        repo_root / 'server' / 'ml',
        repo_root / 'THUNAI.AI' / 'server' / 'ml'
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


# Prefer Drive if mounted; otherwise use local Colab storage.
repo_locations = []
if USE_GOOGLE_DRIVE:
    repo_locations.append(Path('/content/drive/MyDrive/THUNAI.AI'))
repo_locations.append(Path('/content/THUNAI.AI'))

repo_dir = next((p for p in repo_locations if p.exists()), repo_locations[0])

# If the folder exists but does not contain the expected layout, re-clone.
if repo_dir.exists() and find_ml_dir(repo_dir) is None:
    print('Existing folder does not contain server/ml. Re-cloning:', repo_dir)
    shutil.rmtree(repo_dir)

if not repo_dir.exists():
    repo_dir.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', REPO_BRANCH, REPO_URL, str(repo_dir)], check=True)
else:
    print('Repository already exists, reusing it at', repo_dir)

ml_dir = find_ml_dir(repo_dir)

# Fallback: search common roots for any */server/ml path.
if ml_dir is None:
    search_roots = [repo_dir, Path('/content')]
    if USE_GOOGLE_DRIVE:
        search_roots.append(Path('/content/drive/MyDrive'))

    for root in search_roots:
        if not root.exists():
            continue
        for candidate in root.rglob('server/ml'):
            if candidate.is_dir():
                ml_dir = candidate
                break
        if ml_dir is not None:
            break

if ml_dir is None:
    raise FileNotFoundError(
        f'Could not find server/ml inside cloned repository. Checked roots: {search_roots}'
    )

os.chdir(ml_dir)
print('Using ML directory:', ml_dir)

## Step 4: Install dependencies

In [ ]:
import sys
import subprocess

subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', 'pip'], check=True)
subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'torch', 'torchvision', 'pillow', 'numpy',
    'transformers', 'datasets', 'evaluate', 'accelerate'
], check=True)

print('Dependencies installed')

## Step 5: Check GPU

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Step 6: Build dataset splits from Hugging Face

In [ ]:
split_output = Path('/content/data/disease_splits')

cmd = [
    sys.executable, 'prepare_disease_dataset.py',
    '--hf-dataset', HF_DATASET_ID,
    '--hf-split', HF_SPLIT,
    '--hf-image-column', HF_IMAGE_COLUMN,
    '--hf-label-column', HF_LABEL_COLUMN,
    '--output-dir', str(split_output),
    '--val-split', str(VAL_SPLIT),
    '--test-split', str(TEST_SPLIT),
    '--seed', str(SEED)
]

subprocess.run(cmd, check=True)
print('Dataset ready at', split_output)

## Step 7: Train custom and transfer CNN models

In [ ]:
models_output = ml_dir / 'models'
train_dataset_dir = split_output / 'train'

if TRAIN_CUSTOM_CNN:
    custom_cmd = [
        sys.executable, 'train_disease_cnn.py',
        '--dataset', str(train_dataset_dir),
        '--output-dir', str(models_output),
        '--model-type', 'custom-cnn',
        '--epochs', str(CUSTOM_EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--learning-rate', str(LEARNING_RATE_CUSTOM),
        '--val-split', str(VAL_SPLIT),
        '--seed', str(SEED)
    ]
    subprocess.run(custom_cmd, check=True)
    print('Custom CNN training complete')

if TRAIN_TRANSFER_CNN:
    transfer_cmd = [
        sys.executable, 'train_disease_cnn.py',
        '--dataset', str(train_dataset_dir),
        '--output-dir', str(models_output),
        '--model-type', 'pretrained-transfer',
        '--use-imagenet-weights',
        '--epochs', str(TRANSFER_EPOCHS),
        '--batch-size', str(BATCH_SIZE),
        '--learning-rate', str(LEARNING_RATE_TRANSFER),
        '--val-split', str(VAL_SPLIT),
        '--seed', str(SEED)
    ]
    subprocess.run(transfer_cmd, check=True)
    print('Transfer CNN training complete')

## Step 8: Optional Hugging Face model training

In [ ]:
if TRAIN_HF_MODEL:
    hf_cmd = [
        sys.executable, 'train_disease_hf.py',
        '--dataset', str(split_output),
        '--output-dir', str(models_output / 'hf_transfer'),
        '--hf-model', HF_MODEL_ID,
        '--epochs', str(HF_EPOCHS),
        '--batch-size', str(HF_BATCH_SIZE),
        '--learning-rate', str(HF_LEARNING_RATE),
        '--val-split', str(VAL_SPLIT),
        '--seed', str(SEED)
    ]
    subprocess.run(hf_cmd, check=True)
    print('Hugging Face model training complete')
else:
    print('Skipped Hugging Face model training')

## Step 9: Package artifacts

In [ ]:
import zipfile

artifact_candidates = [
    models_output / 'plant_disease_cnn.pt',
    models_output / 'plant_disease_cnn_classes.json',
    models_output / 'plant_disease_cnn_metrics.json',
    models_output / 'plant_disease_transfer.pt',
    models_output / 'plant_disease_transfer_classes.json',
    models_output / 'plant_disease_transfer_metrics.json',
    models_output / 'hf_transfer' / 'hf_training_summary.json'
]

zip_path = Path('/content') / ARTIFACTS_ZIP_NAME
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for file_path in artifact_candidates:
        if file_path.exists():
            zf.write(file_path, arcname=str(file_path.relative_to(models_output.parent)))

print('Artifacts zip created at', zip_path)

if USE_GOOGLE_DRIVE:
    drive_target = Path('/content/drive/MyDrive') / ARTIFACTS_ZIP_NAME
    shutil_cmd = ['cp', str(zip_path), str(drive_target)]
    subprocess.run(shutil_cmd, check=True)
    print('Copied artifacts zip to', drive_target)

## Done

Use the generated artifact files from models folder in THUNAI server/ml/models for inference.